# Execution Lab：CUDA Graph 与 Overlap Scheduling

**目标**：用两个已有 ON/OFF 开关和少量 profiler 标记，亲手确认 mini 的 decode graph 与 overlap loop 实际做了什么。这里不要求重写 graph runner 或调度器。代码改动由你完成；notebook 只给可观察要求、测试负载和阅读入口。


## 从毕业表定位这题

本实验覆盖第 2、7、8、9、10、12 项中的执行侧：prefill/decode 的 shape、attention metadata、decode graph、CPU/GPU overlap 与 ON/OFF ablation。第 7 项的独立 chunk 上限在 [Scheduler Lab](../scheduler/scheduler.ipynb)，第 5 项的树结构在 [KV Cache Lab](../kv_cache/kv_cache.ipynb)。一次 ablation 只能证明在给定模型、输入、并发与硬件上的差异；不能单凭一个数字推断机制。

| 开关 | mini 的现有入口 | 观察什么 |
|---|---|---|
| Graph OFF/ON | `LLM(..., cuda_graph_max_bs=0/4)` | prefill 是否走 eager；decode 是否 replay；batch padding |
| Overlap OFF/ON | 启动前设置 `MINISGL_DISABLE_OVERLAP_SCHEDULING=1/0` | 本轮提交、上一轮结果处理、两条 CUDA stream 的时序 |

两个开关分别对照，不同时改变。`observe.py` 使用 dummy weight 和确定性采样，在每次运行中打印 batch 与输出；可导出 Chrome/Perfetto trace。命令只在 Linux + NVIDIA CUDA、已安装 mini-sglang 的环境运行。


## 先追 mini 的路径

1. `scheduler/scheduler.py`：`run_forever` 在 normal/overlap loop 中如何选批次；`_prepare_batch` 何时产生 positions、page table 映射与 attention metadata；`_process_last_data` 何时同步和回收。
2. `engine/engine.py`：`forward_batch` 怎样选择 eager/graph。
3. `engine/graph.py`：可用性条件、捕获形状、padding、replay 输入缓冲。
4. `attention/base.py` 与实际启用的 backend：prefill/decode 的 metadata 分别在哪里消费。

先画一张两轮 batch 的时序图，标出 scheduler 的 CPU stream、engine 的 GPU stream、batch N 的 forward 与 batch N−1 的结果处理。**不要把“函数返回”直接当成 GPU 已执行完成。** 暂时不用读模型权重、TP 通信、HiCache、FlashAttention kernel 或主仓库的所有 graph backend。


## Phase 0：原样运行，建立输出和形状基线

负载是两条 80 token prompt，`max_extend_tokens=32`，各生成 12 token。运行前预测 prefill/decode batch 的数量、大小和 padding。两条路径分别在独立进程执行；关 overlap 必须在 import mini 之前设环境变量。


In [ ]:
%%bash
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph off --summary /tmp/mini-exec-graph-off.json
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph on --summary /tmp/mini-exec-graph-on.json


**问题**：为什么 prefill 的 `extend_len` 一次可大于 1，而 decode 通常每请求只有 1？graph 能复用的 shape 与动态变化的 page table/positions 如何同时存在？一个 decode batch 的实际人数和 padding 后人数有什么区别？若 `cuda_graph_max_bs=0`，模型是否停止运行？


## Phase 1：小改动，让两条执行路径出现在 profiler

给 mini 的实际执行路径加可读的 profiler range，使下面四个名字能在 trace 里观察到：`lab.eager_forward`、`lab.graph_replay`、`lab.prepare_batch`、`lab.process_result`。要求：Graph OFF 时没有 replay；Graph ON 时 decode 有 replay，prefill 仍有 eager；两种运行都要有 metadata 准备与结果处理。只加观测，不改变计算、调度和同步顺序。

先运行 `--trace` 但不加 `--check-markers`，看缺哪些 range；完成你的改动后加上检查。负载脚本会检查真实 profiler 事件，单纯打印同名字的日志不会通过。

**先答后改**：四段工作分别由哪个 subsystem 持有？`_prepare_batch` 产生的是 GPU kernel 还是 CPU/device metadata 混合工作？图捕获期间的调用与生成期间的 replay 应怎样区分？为什么测量 graph 初始化耗时不能代表稳态 decode？


In [ ]:
%%bash
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph off --trace /tmp/mini-exec-graph-off.trace.json --summary /tmp/mini-exec-graph-off.json --check-markers
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph on --trace /tmp/mini-exec-graph-on.trace.json --summary /tmp/mini-exec-graph-on.json --check-markers


## Phase 2：Overlap Scheduling 的时序与等价性

mini 已有正常循环与 overlap loop。保持 Graph ON 不变，仅切换 overlap。比较同一负载的 token 输出、batch phase 与 profiler trace。预期功能结果一致；实际 CPU/GPU 时间是否重叠，以 trace 为准。请在图上指出：CPU 为 batch N 准备 metadata 时，GPU 可能在做什么？提交 N 之后处理 N−1 结果，哪一步会因 `copy_done` 等待？为什么“有 overlap loop”不保证任意短负载都能测出加速？


In [ ]:
%%bash
MINISGL_DISABLE_OVERLAP_SCHEDULING=1 python lab/execution/observe.py --graph on --trace /tmp/mini-exec-overlap-off.trace.json --summary /tmp/mini-exec-overlap-off.json --check-markers
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph on --trace /tmp/mini-exec-overlap-on.trace.json --summary /tmp/mini-exec-overlap-on.json --check-markers


In [ ]:
import json
from pathlib import Path

def read(name):
    return json.loads(Path('/tmp/mini-exec-' + name + '.json').read_text())

for a, b in [('graph-off', 'graph-on'), ('overlap-off', 'overlap-on')]:
    left, right = read(a), read(b)
    assert left['token_ids'] == right['token_ids'], (a, b, 'different outputs')
    assert [x['phase'] for x in left['batches']] == [x['phase'] for x in right['batches']], (a, b, 'different phases')
    print(a, b, 'outputs and phases match')


## Phase 3：ON/OFF ablation + profiler/benchmark

Profiler 用来证明走了哪条路径、哪里等待；性能数字用关闭 profiler 的重复运行。下面每个命令会先 warmup，再用不同 token 前缀跑五轮，输出单轮毫秒数与中位数。分别做 Graph 对照和 Overlap 对照，其他参数、模型、硬件保持一致。至少报告原始五个值、median、GPU 型号、图捕获是否计入测量、是否有前缀缓存命中；不要只写“快了 x%”。如果两者差异落在噪声范围内，也要保留结果并解释当前负载缺少什么。


In [ ]:
%%bash
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph off --bench --summary /tmp/mini-exec-bench-graph-off.json
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph on --bench --summary /tmp/mini-exec-bench-graph-on.json
MINISGL_DISABLE_OVERLAP_SCHEDULING=1 python lab/execution/observe.py --graph on --bench --summary /tmp/mini-exec-bench-overlap-off.json
MINISGL_DISABLE_OVERLAP_SCHEDULING=0 python lab/execution/observe.py --graph on --bench --summary /tmp/mini-exec-bench-overlap-on.json


## 对照主仓库与交付

阅读 [主仓库 Scheduler](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/managers/scheduler.py) 的正常/overlap event loop、[PrefillAdder](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/managers/schedule_policy.py) 的输入/chunk 预算、[ModelRunner](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/model_executor/model_runner.py) 的 graph 决策。主仓库支持的 prefill graph、复杂 graph backend、分布式调度、PD 分离和抢占超出本实验；写出 mini 无法直接照搬的两个契约即可，不要求移植。

交付：两张 trace 局部截图（Graph OFF/ON；Overlap OFF/ON）、一张带四组原始样本的 ablation 表、两轮 batch 时序图和三个解释：为什么 graph 主要优化 decode、overlap 的两端各是什么、Chunked Prefill 改动为什么会同时影响 TTFT 与 ITL。不要把 profiler 的高开销运行当成性能结论。
